# Sparse AI V3+ Frontier Suite — H100 / A100 (80GB) Execution Notebook
**Synthesis of V1–V3 Ultrametric Routing, Fused Triton Block-List Autograd, Multi-Prime GQA Forests, & Surprisal-Gated Adèlic Cache**

### Supported 2025–2026 Frontier Models (Single 80GB H100/A100):
- **Mid-Size Long-Context Sweet Spot (16–29 GB in `bfloat16`, 32K–64K context headroom)**:
  - `Qwen/Qwen3-14B` (40L, 5:1 GQA, `head_dim=128`, QK-Norm)
  - `Qwen/Qwen3-8B` (36L, 4:1 GQA, `head_dim=128`, QK-Norm)
  - `Qwen/Qwen3.5-9B` (3:1 Hybrid GatedDeltaNet + Gated Full Attention)
  - `meta-llama/Meta-Llama-3.1-8B-Instruct` (32L, 4:1 GQA, baseline reference)
- **Flagship 24B–32B Dense / Hybrid / MoE (48–65 GB in `bfloat16` or ~16 GB in 4-bit NF4)**:
  - `Qwen/Qwen3-32B` (64L pure full GQA, `head_dim=128`, QK-Norm)
  - `Qwen/Qwen3.6-27B` / `sneedjak/Adelic-Qwen3.6-27B-Topology` (64L 3:1 Hybrid: only 16 full-attention layers!)
  - `google/gemma-4-31B-it` / `google/gemma-4-26B-A4B-it` (60L 5:1 Sliding/Global: only 10 global layers need patching!)
  - `mistralai/Mistral-Small-3.2-24B-Instruct-2506` (40L pure GQA)

In [ ]:
# 1. Environment & HF_TOKEN Setup from Colab Secrets
import os
import sys
import subprocess

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
        print('[Auth] Loaded HF_TOKEN from Colab Secrets.')
except Exception as e:
    print(f'[Auth] Not running in Colab or HF_TOKEN secret not found ({e}); checking environment.')

REPO_URL = os.environ.get('SPARSE_AI_REPO', 'https://github.com/sneed-and-feed/sparse-ai.git')
if not os.path.exists('src/llama_surgery') and not os.path.exists('../src/llama_surgery'):
    subprocess.run(['git', 'clone', REPO_URL, 'sparse-ai'], check=False)
    if os.path.exists('sparse-ai'):
        os.chdir('sparse-ai')
elif os.path.exists('../src/llama_surgery') and not os.path.exists('src/llama_surgery'):
    os.chdir('..')

if os.path.exists('.git'):
    subprocess.run(['git', 'pull', '--ff-only'], check=False)

sys.path.insert(0, os.path.abspath('src'))
sys.path.insert(0, os.path.abspath('experiments'))

import torch
import triton
print(f'PyTorch: {torch.__version__} | Triton: {triton.__version__} | CUDA: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
if torch.cuda.is_available():
    print(f'VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

## 2. Track 2 Verification: Fused Triton Block-List Builder (`<12 µs`) & Sparse Backward Autograd
Benchmarks `_build_block_lists_kernel` + `block_list_attention` (forward + sparse backward with STE `soft_block_bias`) against PyTorch SDPA across $N \in [4096, 8192, 16384, 32768]$.

In [ ]:
import time
import torch.nn.functional as F
from ultrametric.block_list import build_block_lists, block_list_attention

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.bfloat16 if (device == 'cuda' and torch.cuda.is_bf16_supported()) else torch.float16

Z, H, D, RB, TD = 1, 32, 128, 128, 6
req_depth = 4  # p=2 -> 2^-4 = 6.25% theoretical block density

print(f'{"SeqLen":>8} | {"Active%":>8} | {"ListBuild(ms)":>14} | {"SparseFwd(ms)":>14} | {"SparseBwd(ms)":>14} | {"SDPA_FwdBwd(ms)":>16} | {"Speedup":>8}')
print('-' * 98)

for N in [4096, 8192, 16384, 32768]:
    NB = N // RB
    routes = torch.randint(0, 2, (Z, H, NB, TD), device=device, dtype=torch.int32)
    q = torch.randn(Z, H, N, D, device=device, dtype=dtype, requires_grad=True)
    k = torch.randn(Z, H, N, D, device=device, dtype=dtype, requires_grad=True)
    v = torch.randn(Z, H, N, D, device=device, dtype=dtype, requires_grad=True)
    do = torch.randn_like(q)

    # Warmup
    bl = build_block_lists(routes, req_depth, is_causal=True, sink_block=True, local_blocks=1, arity=2)
    soft_bias = torch.zeros((Z, H, NB, bl.max_count), device=device, dtype=torch.float32, requires_grad=True)
    out = block_list_attention(q, k, v, bl, route_block=RB, is_causal=True, soft_block_bias=soft_bias)
    out.backward(do)
    torch.cuda.synchronize()

    # Measure Fused List Builder
    t0 = time.perf_counter()
    for _ in range(20):
        bl = build_block_lists(routes, req_depth, is_causal=True, sink_block=True, local_blocks=1, arity=2)
    torch.cuda.synchronize()
    t_list = (time.perf_counter() - t0) * 1000 / 20

    # Measure Sparse Fwd + Bwd
    q.grad = k.grad = v.grad = soft_bias.grad = None
    t0 = time.perf_counter()
    out = block_list_attention(q, k, v, bl, route_block=RB, is_causal=True, soft_block_bias=soft_bias)
    torch.cuda.synchronize()
    t_fwd = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    out.backward(do)
    torch.cuda.synchronize()
    t_bwd = (time.perf_counter() - t0) * 1000

    # Measure Dense SDPA Fwd + Bwd
    q.grad = k.grad = v.grad = None
    t0 = time.perf_counter()
    out_sdpa = F.scaled_dot_product_attention(q, k, v, is_causal=True)
    out_sdpa.backward(do)
    torch.cuda.synchronize()
    t_sdpa = (time.perf_counter() - t0) * 1000

    density = 100.0 * bl.counts.float().sum().item() / (Z * H * NB * (NB + 1) / 2)
    speedup = t_sdpa / (t_list + t_fwd + t_bwd)
    print(f'{N:8d} | {density:7.2f}% | {t_list:14.3f} | {t_fwd:14.3f} | {t_bwd:14.3f} | {t_sdpa:16.3f} | {speedup:7.2f}x')

## 3. Select Frontier Model & Inject V3+ Surgical Ultrametric Topology
Configure `MODEL_ID` below. By default, we enable:
- `qk_mode=True` (Post-RoPE valid-length masked mean + peak-norm block pooling — **fixes R1 & R5**)
- `prime_arities=[2, 3]` (Multi-Prime Adèlic GQA head groups — **fixes R2**)
- `swin_shift=True` (Odd-layer cyclic block shifting — **fixes R3**)
- `distill_kl=True` ($O(NB^2)$ Tiled KL Distillation + Dual Lagrangian density control)
- `route_stride=2` (Cross-layer routing reuse)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from llama_surgery import inject_surgery
from sweep_topological_niah import warmup_router_wikitext

# Pick any frontier model that fits your runtime:
# - 'Qwen/Qwen3-14B' or 'Qwen/Qwen3-8B'
# - 'Qwen/Qwen3-32B' or 'Qwen/Qwen3.6-27B'
# - 'google/gemma-4-31B-it' (set ONLY_GLOBAL_LAYERS = True)
# - 'mistralai/Mistral-Small-3.2-24B-Instruct-2506'
# - 'meta-llama/Meta-Llama-3.1-8B-Instruct'
MODEL_ID = 'Qwen/Qwen3-14B'
ONLY_GLOBAL_LAYERS = 'gemma' in MODEL_ID.lower()
LOAD_IN_4BIT = False  # Set True for 70B models or 32B w/ 32K+ training windows

hf_token = os.environ.get('HF_TOKEN')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

load_kwargs = dict(device_map='cuda', token=hf_token)
if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    load_kwargs['quantization_config'] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type='nf4'
    )
else:
    load_kwargs['torch_dtype'] = torch.bfloat16

print(f'Loading {MODEL_ID}...')
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)

model = inject_surgery(
    model,
    tree_depth=5,
    arity=2,
    preserve_sinks=True,
    init_mode='collapse',
    tree_mode=True,
    qk_mode=True,
    prime_arities=[2, 3],
    swin_shift=True,
    route_stride=2,
    distill_kl=True,
    target_density=0.25,
    only_global_layers=ONLY_GLOBAL_LAYERS,
)
model.config.surgical_attention_backend = 'block_list'
print('Surgical injection complete!')

## 4. Long-Context Router Training (4K–8K Tokens via Sparse Block-List Autograd + KL-Lagrangian)
Trains only the lightweight `DynamicTopologyRouter` parameters on concatenated WikiText documents at `max_length=4096`.

In [ ]:
warmup_router_wikitext(
    model,
    tokenizer,
    steps=100,
    lr=2e-3,
    device='cuda',
    max_length=4096,
)
model.eval()
model.config.use_cache = True

## 5. Track 3 Verification: `SurprisalAdelicCache` (Outlier Shield + Medoid Condenser) & Self-Speculative Decoding
Tests long-context needle retention under a fixed physical KV-cache ceiling (`max_capacity=1024`) and benchmarks `SpeculativeSurgicalGenerator`.

In [ ]:
from llama_surgery import SurprisalAdelicCache, SpeculativeSurgicalGenerator

# 5A. SurprisalAdelicCache Needle-In-A-Haystack Test under 512-token physical ceiling
cache = SurprisalAdelicCache(max_capacity=512, local_window=128, outlier_budget=128, lambda_decay=0.05)
needle_prompt = (
    'System Log Entry #0041: The secret vault override code is 9481-OMEGA. '
    + ('The background telemetry system reports nominal pressure across all sectors. ' * 120)
    + '\nQuestion: What is the secret vault override code?\nAnswer:'
)
inputs = tokenizer(needle_prompt, return_tensors='pt').to('cuda')
with torch.no_grad():
    model.config.surgical_req_depth = 0
    gen_ids = model.generate(**inputs, past_key_values=cache, max_new_tokens=24, do_sample=False)
ans = tokenizer.decode(gen_ids[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f'[SurprisalAdelicCache] Prompt len: {inputs["input_ids"].shape[1]} -> Physical KV len: {cache.get_physical_seq_length(0)} | Answer: {ans.strip()}')

# 5B. Self-Speculative Decoding (Draft r=4 -> Verify r=0)
spec_gen = SpeculativeSurgicalGenerator(model, draft_req_depth=4, verify_req_depth=0, gamma=5)
spec_ids, stats = spec_gen.generate(inputs['input_ids'], max_new_tokens=32, temperature=0.0, return_stats=True)
print(f'[Self-Speculative Decode] Acceptance rate: {stats["acceptance_rate"]:.2%} | Tokens/verify-step: {stats["tokens_per_step"]:.2f}')
print('Generated:', tokenizer.decode(spec_ids[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip())

## 6. Full LongBench QASPER Evaluation (200 Samples)
Runs the updated `experiments/eval_qasper_router.py` pipeline with `--qk_mode --prime_arities 2,3 --swin_shift --distill_kl --attention_backend block_list`.

In [ ]:
!python experiments/eval_qasper_router.py \
    --model_id {MODEL_ID} \
    --depths 2,3,4 \
    --max_samples 200 \
    --train_steps 100 \
    --train_max_length 4096 \
    --tree_mode \
    --qk_mode \
    --prime_arities 2,3 \
    --swin_shift \
    --route_stride 2 \
    --distill_kl \
    --target_density 0.25 \
    --attention_backend block_list